In [2]:
import pandas as pd

df = pd.read_csv("/content/yellow_tripdata_2025-10.csv")

print("Data loaded successfully!")

print(df.head())

print("Dataset shape:", df.shape)

/tmp/ipykernel_439/1726671060.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/yellow_tripdata_2025-10.csv")


Data loaded successfully!
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2025-10-01 00:15:32   2025-10-01 01:04:03              1.0   
1         7  2025-10-01 00:00:08   2025-10-01 00:00:08              1.0   
2         2  2025-10-01 00:08:54   2025-10-01 00:14:44              1.0   
3         1  2025-10-01 00:58:48   2025-10-01 01:04:40              1.0   
4         2  2025-10-01 00:39:51   2025-10-01 00:49:40              1.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0          17.20         2.0                  N           132           107   
1           5.00         1.0                  N           107           225   
2           2.75         1.0                  N           263           229   
3           1.30         1.0                  N           211           231   
4           2.88         1.0                  N           230           151   

   payment_type  fare_amount  extra  mta_tax  ti

In [3]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])

df["date"] = df["tpep_pickup_datetime"].dt.date
df["hour"] = df["tpep_pickup_datetime"].dt.hour

demand_df = df.groupby(
    ["date", "hour", "PULocationID"]
).size().reset_index(name="demand")

print(demand_df.head())

         date  hour  PULocationID  demand
0  2025-09-30    22           164       1
1  2025-09-30    23            79       1
2  2025-09-30    23           107       1
3  2025-09-30    23           140       1
4  2025-09-30    23           142       1


In [4]:
demand_df.to_csv("hourly_demand_dataset.csv", index=False)

In [5]:
# 时间特征
demand_df["timestamp"] = pd.to_datetime(demand_df["date"]) + pd.to_timedelta(demand_df["hour"], unit="h")

demand_df["day_of_week"] = demand_df["timestamp"].dt.dayofweek

demand_df["is_weekend"] = demand_df["day_of_week"].isin([5,6]).astype(int)

# 按 zone 排序
demand_df = demand_df.sort_values(["PULocationID","timestamp"])

# 上一小时需求
demand_df["lag_1"] = demand_df.groupby("PULocationID")["demand"].shift(1)

# 过去3小时平均需求
demand_df["rolling_mean_3"] = demand_df.groupby("PULocationID")["demand"].shift(1).rolling(3).mean()

# 预测目标：下一小时需求
demand_df["target"] = demand_df.groupby("PULocationID")["demand"].shift(-1)

# 删除空值
model_df = demand_df.dropna()

print(model_df.head())

            date  hour  PULocationID  demand           timestamp  day_of_week  \
2395  2025-10-01    15             1       2 2025-10-01 15:00:00            2   
2580  2025-10-01    16             1       2 2025-10-01 16:00:00            2   
3081  2025-10-01    19             1       1 2025-10-01 19:00:00            2   
3381  2025-10-01    21             1       3 2025-10-01 21:00:00            2   
4492  2025-10-02     6             1       1 2025-10-02 06:00:00            3   

      is_weekend  lag_1  rolling_mean_3  target  
2395           0    1.0        1.000000     2.0  
2580           0    2.0        1.333333     1.0  
3081           0    2.0        1.666667     3.0  
3381           0    1.0        1.666667     1.0  
4492           0    3.0        2.000000     1.0  


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "lag_1",
    "rolling_mean_3"
]

X = model_df[features]
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

MAE: 9.65119453681349
RMSE: 23.362366503607134


In [8]:
result = X_test.copy()

result["actual_demand"] = y_test
result["predicted_demand"] = pred

result["imbalance"] = result["predicted_demand"] - result["actual_demand"]

print(result.head())

        hour  day_of_week  is_weekend  lag_1  rolling_mean_3  actual_demand  \
5898      13            3           0    1.0        3.666667            2.0   
113515     8            3           0    4.0        3.000000            2.0   
78694      7            1           0    7.0        3.333333            8.0   
99296     13            6           1    5.0        4.333333            1.0   
75495     10            0           0    4.0        2.000000            1.0   

        predicted_demand  imbalance  
5898            2.477000   0.477000  
113515          5.872258   3.872258  
78694           9.639714   1.639714  
99296           3.802214   2.802214  
75495           1.720457   0.720457  


In [9]:
def classify(x):

    if x > 5:
        return "shortage"

    elif x < -5:
        return "surplus"

    else:
        return "balanced"

result["status"] = result["imbalance"].apply(classify)

print(result.head())

        hour  day_of_week  is_weekend  lag_1  rolling_mean_3  actual_demand  \
5898      13            3           0    1.0        3.666667            2.0   
113515     8            3           0    4.0        3.000000            2.0   
78694      7            1           0    7.0        3.333333            8.0   
99296     13            6           1    5.0        4.333333            1.0   
75495     10            0           0    4.0        2.000000            1.0   

        predicted_demand  imbalance    status  
5898            2.477000   0.477000  balanced  
113515          5.872258   3.872258  balanced  
78694           9.639714   1.639714  balanced  
99296           3.802214   2.802214  balanced  
75495           1.720457   0.720457  balanced  


In [10]:
result["status"].value_counts()

,count
status,
balanced,16003
shortage,4287
surplus,3651


In [11]:
shortage_count = len(result[result["status"]=="shortage"])
surplus_count = len(result[result["status"]=="surplus"])

vehicles_moved = min(shortage_count, surplus_count)

print("Shortage zones:", shortage_count)
print("Surplus zones:", surplus_count)
print("Vehicles rebalanced:", vehicles_moved)

Shortage zones: 4287
Surplus zones: 3651
Vehicles rebalanced: 3651
